# HomeMatch Application
This notebook fulfills the requirements for the HomeMatch project.

## Step 1: Setup
Please put your Vocareum OpenAI API key in this cell before running it.

In [1]:
import os
import json
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

# IMPORTANT: Set your OpenAI API key here
os.environ["OPENAI_API_KEY"] = "voc-42685436912667751077556a13ced6318000.01472501"
# Vocareum OpenAI API Base
os.environ["OPENAI_API_BASE"] = "https://openai.vocareum.com/v1"

# Using the Vocareum compatible configuration
llm = ChatOpenAI(temperature=0.7, model="gpt-3.5-turbo")
parser = StrOutputParser()


## Step 2: Generating Real Estate Listings
We use LangChain and ChatOpenAI to generate 10 listings.

In [2]:
listing_template = '''
Generate a realistic real estate listing. Provide the following details:
Neighborhood, Price, Bedrooms, Bathrooms, House Size (sqft), Description, and Neighborhood Description.

Make sure it is unique. Do not repeat previous examples.
Return the output strictly in the following JSON format:
{{
    "Neighborhood": "...",
    "Price": "...",
    "Bedrooms": 0,
    "Bathrooms": 0,
    "House Size": 0,
    "Description": "...",
    "Neighborhood Description": "..."
}}
'''
prompt = PromptTemplate.from_template(listing_template)
chain = prompt | llm | parser

listings = []
print("Generating 15 listings... This may take a minute.")
for i in range(15):
    try:
        response = chain.invoke({})
        listing_data = json.loads(response)
        listings.append(listing_data)
        print(f"Generated listing {i+1}")
    except Exception as e:
        print(f"Failed to generate/parse listing {i+1}: {e}")

# Save to Listings.txt
with open("Listings.txt", "w", encoding="utf-8") as f:
    for i, listing in enumerate(listings):
        f.write(f"--- Listing {i+1} ---\n")
        f.write(f"Neighborhood: {listing.get('Neighborhood', '')}\n")
        f.write(f"Price: {listing.get('Price', '')}\n")
        f.write(f"Bedrooms: {listing.get('Bedrooms', '')}\n")
        f.write(f"Bathrooms: {listing.get('Bathrooms', '')}\n")
        f.write(f"House Size: {listing.get('House Size', '')} sqft\n")
        f.write(f"Description: {listing.get('Description', '')}\n")
        f.write(f"Neighborhood Description: {listing.get('Neighborhood Description', '')}\n\n")

print("Saved listings to Listings.txt")


Generating 15 listings... This may take a minute.
Generated listing 1
Generated listing 2
Generated listing 3
Generated listing 4
Generated listing 5
Generated listing 6
Generated listing 7
Generated listing 8
Generated listing 9
Generated listing 10
Generated listing 11
Generated listing 12
Generated listing 13
Generated listing 14
Generated listing 15
Saved listings to Listings.txt


## Step 3: Storing Listings in a Vector Database
We create embeddings and store them in ChromaDB.

In [3]:
documents = []
metadatas = []
for listing in listings:
    content = f"Neighborhood: {listing['Neighborhood']}\nPrice: {listing['Price']}\nBedrooms: {listing['Bedrooms']}\nBathrooms: {listing['Bathrooms']}\nHouse Size: {listing['House Size']} sqft\nDescription: {listing['Description']}\nNeighborhood Description: {listing['Neighborhood Description']}"
    documents.append(content)
    
    metadatas.append({
        "Neighborhood": listing["Neighborhood"],
        "Price": str(listing["Price"]),
        "Bedrooms": listing["Bedrooms"],
        "Bathrooms": listing["Bathrooms"],
        "House Size": listing["House Size"]
    })

embeddings = OpenAIEmbeddings()
vector_db = Chroma.from_texts(texts=documents, embedding=embeddings, metadatas=metadatas, collection_name="real_estate_listings")
print("Vector database created with listings.")


Vector database created with listings.


## Step 4 & 5: Building User Preference Interface & Semantic Search
We parse buyer preferences and retrieve top matches from the DB.

In [4]:
buyer_preferences = [
    "A comfortable three-bedroom house with a spacious kitchen and a cozy living room.",
    "A quiet neighborhood, good local schools, and convenient shopping options.",
    "A backyard for gardening, a two-car garage, and a modern, energy-efficient heating system.",
    "Easy access to a reliable bus line, proximity to a major highway, and bike-friendly roads.",
    "A balance between suburban tranquility and access to urban amenities like restaurants and theaters."
]

query = " ".join(buyer_preferences)
print("Buyer query:\n", query)

results = vector_db.similarity_search(query, k=3)
print(f"\nFound {len(results)} matching listings.\n")
for i, res in enumerate(results):
    print(f"--- Match {i+1} ---")
    print(res.page_content)
    print()


Buyer query:
 A comfortable three-bedroom house with a spacious kitchen and a cozy living room. A quiet neighborhood, good local schools, and convenient shopping options. A backyard for gardening, a two-car garage, and a modern, energy-efficient heating system. Easy access to a reliable bus line, proximity to a major highway, and bike-friendly roads. A balance between suburban tranquility and access to urban amenities like restaurants and theaters.

Found 3 matching listings.

--- Match 1 ---
Neighborhood: Historic Downtown District
Price: $750,000
Bedrooms: 3
Bathrooms: 2
House Size: 2100 sqft
Description: Charming Victorian home with modern updates located in the heart of the Historic Downtown District. This property features a spacious kitchen, hardwood floors, and a beautifully landscaped backyard with a patio perfect for entertaining.
Neighborhood Description: The Historic Downtown District is known for its picturesque streets lined with historic homes, boutique shops, and top-rat

## Step 6: Personalizing Listing Descriptions
We use an LLM to augment the description according to buyer preferences without changing facts.

In [5]:
augment_template = '''
You are a helpful real estate agent. You have a property listing and a buyer's preferences.
Rewrite the description of the property to emphasize how it meets the buyer's preferences.
Do NOT change any factual information about the property (like Price, Bedrooms, Bathrooms, House Size, or Neighborhood).
Make it appealing and tailored to the buyer.

Buyer Preferences:
{preferences}

Property Listing:
{listing}

Personalized Description:
'''
augment_prompt = PromptTemplate.from_template(augment_template)
augment_chain = augment_prompt | llm | parser

for i, res in enumerate(results):
    print(f"=== Personalized Listing {i+1} ===")
    
    meta = res.metadata
    print(f"Neighborhood: {meta.get('Neighborhood')}")
    print(f"Price: {meta.get('Price')}")
    print(f"Bedrooms: {meta.get('Bedrooms')}")
    print(f"Bathrooms: {meta.get('Bathrooms')}")
    print(f"House Size: {meta.get('House Size')} sqft\n")
    
    personalized_desc = augment_chain.invoke({
        "preferences": query,
        "listing": res.page_content
    })
    print(f"Personalized Description:\n{personalized_desc}\n")


=== Personalized Listing 1 ===
Neighborhood: Historic Downtown District
Price: $750,000
Bedrooms: 3
Bathrooms: 2
House Size: 2100 sqft

Personalized Description:
Welcome to your dream home located in the heart of the charming Historic Downtown District! This cozy three-bedroom Victorian home has everything you've been looking for. The spacious kitchen is perfect for preparing delicious meals, while the cozy living room is ideal for relaxing with loved ones. 

Situated in a quiet neighborhood with top-rated schools and convenient shopping options, this property also boasts a beautifully landscaped backyard for all your gardening needs. The two-car garage provides ample storage space, and the modern, energy-efficient heating system ensures your comfort year-round. 

With easy access to a reliable bus line, proximity to a major highway, and bike-friendly roads, commuting is a breeze. Plus, you'll love the balance between suburban tranquility and urban amenities like restaurants and theate